## Section 9 Block 1 — Tuned classical classifiers

In [ ]:
# SECTION 9 — AXIS 2: CLASSIFIER & TRAINING DESIGN
# BLOCK 1 — PART A (A2a, A2b, A2c)

def evaluate_axis2_10fold(cv_df, test_texts, y_test, text_col="q_ctx"):
    axis2_results = {}
    axis2_preds_dict = {}

    # Shared final features for all word+char models
    X_cv_wc, [X_test_wc], word_vec_final, char_vec_final = build_word_char_features(
        cv_df[text_col].tolist(), [test_texts]
    )

    # A2a: Tuned LR (word+char TF-IDF)
    print("A2a — Tuned Logistic Regression (word+char TF-IDF)")
    fold_f1s_lr = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr_texts = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_va_texts = cv_df.loc[val_mask, text_col].tolist()
        y_va = cv_df.loc[val_mask, "label_id"].values

        X_tr_wc, [X_va_wc], _, _ = build_word_char_features(X_tr_texts, [X_va_texts])

        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        lr_grid = GridSearchCV(
            LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                solver="liblinear",
                random_state=RANDOM_SEED
            ),
            param_grid={"C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
            cv=inner_cv,
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        lr_grid.fit(X_tr_wc, y_tr)
        val_preds = lr_grid.predict(X_va_wc)
        fold_f1s_lr.append(f1_score(y_va, val_preds, average="macro"))

    lr_final = GridSearchCV(
        LogisticRegression(
            max_iter=4000,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_SEED
        ),
        param_grid={"C": [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    lr_final.fit(X_cv_wc, y_cv)
    pred_lr = lr_final.predict(X_test_wc)

    print(f"  Best C              : {lr_final.best_params_['C']}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_lr):.4f} ± {np.std(fold_f1s_lr):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_lr):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_lr, average='macro'):.4f}")
    print(classification_report(y_test, pred_lr, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2a: LR word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_lr):.4f} ± {np.std(fold_f1s_lr):.4f}",
        "Test Acc": accuracy_score(y_test, pred_lr),
        "Test Macro-F1": f1_score(y_test, pred_lr, average="macro")
    }
    axis2_preds_dict["A2a_LR_wordchar"] = pred_lr

    # A2b: Calibrated Linear SVM + maybe-threshold tuning 
    print("\nA2b — Calibrated Linear SVM + maybe-threshold tuning")
    fold_f1s_svm = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        outer_train_df = cv_df.loc[train_mask].reset_index(drop=True)
        outer_val_df = cv_df.loc[val_mask].reset_index(drop=True)

        X_outer_train = outer_train_df[text_col].tolist()
        y_outer_train = outer_train_df["label_id"].values
        X_outer_val = outer_val_df[text_col].tolist()
        y_outer_val = outer_val_df["label_id"].values

        inner_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        inner_train_idx, inner_dev_idx = next(inner_splitter.split(X_outer_train, y_outer_train))

        X_inner_train = [X_outer_train[i] for i in inner_train_idx]
        y_inner_train = y_outer_train[inner_train_idx]
        X_inner_dev = [X_outer_train[i] for i in inner_dev_idx]
        y_inner_dev = y_outer_train[inner_dev_idx]

        X_inner_wc, [X_dev_wc, X_outer_val_wc], _, _ = build_word_char_features(
            X_inner_train, [X_inner_dev, X_outer_val]
        )

        svm_grid = GridSearchCV(
            LinearSVC(class_weight="balanced", random_state=RANDOM_SEED, max_iter=10000),
            param_grid={"C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]},
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        svm_grid.fit(X_inner_wc, y_inner_train)

        cal_svm_fold = CalibratedClassifierCV(
            svm_grid.best_estimator_,
            method="sigmoid",
            cv=5
        )
        cal_svm_fold.fit(X_inner_wc, y_inner_train)

        dev_probs = cal_svm_fold.predict_proba(X_dev_wc)
        best_thr_fold, best_dev_f1_fold = None, -1.0

        for thr in np.arange(0.25, 0.61, 0.025):
            dev_pred_thr = maybe_threshold_predict(dev_probs, maybe_threshold=thr, maybe_class_id=1)
            dev_f1 = f1_score(y_inner_dev, dev_pred_thr, average="macro")
            if dev_f1 > best_dev_f1_fold:
                best_dev_f1_fold = dev_f1
                best_thr_fold = float(thr)

        outer_val_probs = cal_svm_fold.predict_proba(X_outer_val_wc)
        outer_val_preds = maybe_threshold_predict(
            outer_val_probs,
            maybe_threshold=best_thr_fold,
            maybe_class_id=1
        )
        fold_f1s_svm.append(f1_score(y_outer_val, outer_val_preds, average="macro"))

    dev_fold_mask = cv_df["fold"] == 0
    train_no_dev_mask = ~dev_fold_mask

    X_tr_texts_svm = cv_df.loc[train_no_dev_mask, text_col].tolist()
    y_tr_svm = cv_df.loc[train_no_dev_mask, "label_id"].values
    X_dev_texts_svm = cv_df.loc[dev_fold_mask, text_col].tolist()
    y_dev_svm = cv_df.loc[dev_fold_mask, "label_id"].values

    X_tr_wc_s, [X_dev_wc_s, X_test_wc_s], _, _ = build_word_char_features(
        X_tr_texts_svm, [X_dev_texts_svm, test_texts]
    )

    svm_final = GridSearchCV(
        LinearSVC(class_weight="balanced", random_state=RANDOM_SEED, max_iter=10000),
        param_grid={"C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    svm_final.fit(X_tr_wc_s, y_tr_svm)

    cal_svm = CalibratedClassifierCV(
        svm_final.best_estimator_,
        method="sigmoid",
        cv=5
    )
    cal_svm.fit(X_tr_wc_s, y_tr_svm)

    dev_probs = cal_svm.predict_proba(X_dev_wc_s)
    best_thr, best_dev_f1 = None, -1.0
    for thr in np.arange(0.25, 0.61, 0.025):
        dev_pred_thr = maybe_threshold_predict(dev_probs, maybe_threshold=thr, maybe_class_id=1)
        dev_f1 = f1_score(y_dev_svm, dev_pred_thr, average="macro")
        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_thr = float(thr)

    test_probs_svm = cal_svm.predict_proba(X_test_wc_s)
    pred_svm_thr = maybe_threshold_predict(
        test_probs_svm,
        maybe_threshold=best_thr,
        maybe_class_id=1
    )

    print(f"  Best C              : {svm_final.best_params_['C']}")
    print(f"  Best dev threshold  : {best_thr:.3f}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_svm):.4f} ± {np.std(fold_f1s_svm):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_svm_thr):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_svm_thr, average='macro'):.4f}")
    print(classification_report(y_test, pred_svm_thr, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2b: Cal-SVM word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_svm):.4f} ± {np.std(fold_f1s_svm):.4f}",
        "Test Acc": accuracy_score(y_test, pred_svm_thr),
        "Test Macro-F1": f1_score(y_test, pred_svm_thr, average="macro")
    }
    axis2_preds_dict["A2b_CalSVM_wordchar"] = pred_svm_thr

    # A2c: Multinomial Naive Bayes 
    print("\nA2c — Multinomial Naive Bayes (word+char TF-IDF)")
    fold_f1s_nb = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr_texts = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_va_texts = cv_df.loc[val_mask, text_col].tolist()
        y_va = cv_df.loc[val_mask, "label_id"].values

        X_tr_wc, [X_va_wc], _, _ = build_word_char_features(X_tr_texts, [X_va_texts])

        nb_grid = GridSearchCV(
            MultinomialNB(),
            param_grid={"alpha": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]},
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
            scoring="f1_macro",
            refit=True,
            n_jobs=-1
        )
        nb_grid.fit(X_tr_wc, y_tr)
        val_preds = nb_grid.predict(X_va_wc)
        fold_f1s_nb.append(f1_score(y_va, val_preds, average="macro"))

    nb_final = GridSearchCV(
        MultinomialNB(),
        param_grid={"alpha": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]},
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        scoring="f1_macro",
        refit=True,
        n_jobs=-1
    )
    nb_final.fit(X_cv_wc, y_cv)
    pred_nb = nb_final.predict(X_test_wc)

    print(f"  Best alpha          : {nb_final.best_params_['alpha']}")
    print(f"  10-Fold CV Macro-F1 : {np.mean(fold_f1s_nb):.4f} ± {np.std(fold_f1s_nb):.4f}")
    print(f"  Test Acc            : {accuracy_score(y_test, pred_nb):.4f}")
    print(f"  Test Macro-F1       : {f1_score(y_test, pred_nb, average='macro'):.4f}")
    print(classification_report(y_test, pred_nb, target_names=["no", "maybe", "yes"], zero_division=0))

    axis2_results["A2c: MNB word+char"] = {
        "CV Macro-F1": f"{np.mean(fold_f1s_nb):.4f} ± {np.std(fold_f1s_nb):.4f}",
        "Test Acc": accuracy_score(y_test, pred_nb),
        "Test Macro-F1": f1_score(y_test, pred_nb, average="macro")
    }
    axis2_preds_dict["A2c_MNB_wordchar"] = pred_nb

    return (
        axis2_results,
        axis2_preds_dict,
        lr_final,
        svm_final,
        nb_final,
        word_vec_final,
        char_vec_final,
        X_cv_wc,
        X_test_wc
    )


axis2_results, axis2_preds_dict, lr_final, svm_final, nb_final, word_vec_final, char_vec_final, X_cv_wc, X_test_wc = \
    evaluate_axis2_10fold(cv_df, X_test_qctx, y_test, text_col="q_ctx")

for k, v in axis2_results.items():
    all_results[k] = v
for k, v in axis2_preds_dict.items():
    all_preds[k] = v

## Section 9 Block 2 — SMOTE and soft voting


In [ ]:

# SECTION 9 — BLOCK 2 — A2d and A2e

import sys
import subprocess
import numpy as np

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    print("imbalanced-learn not found. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE


required_vars = [
    "cv_df", "X_cv_wc", "X_test_wc", "y_cv", "y_test",
    "RANDOM_SEED", "build_word_char_features",
    "axis2_results", "axis2_preds_dict", "all_results", "all_preds",
    "svm_final", "lr_final", "nb_final"
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(
        "These required variables are missing. Run the earlier cells first: "
        + ", ".join(missing_vars)
    )


# A2d: SMOTE + Logistic Regression
print("\nA2d — SMOTE + Logistic Regression (leakage-free, inside 10-fold CV)")

fold_f1s_smote = []

for fold_idx in range(10):
    val_mask = cv_df["fold"] == fold_idx
    train_mask = ~val_mask

    X_tr_texts = cv_df.loc[train_mask, "q_ctx"].tolist()
    y_tr = cv_df.loc[train_mask, "label_id"].values

    X_va_texts = cv_df.loc[val_mask, "q_ctx"].tolist()
    y_va = cv_df.loc[val_mask, "label_id"].values

    X_tr_wc, [X_va_wc], _, _ = build_word_char_features(
        X_tr_texts,
        [X_va_texts]
    )

    class_counts = np.bincount(y_tr)
    min_class_count = class_counts[class_counts > 0].min()
    smote_k = min(5, min_class_count - 1)

    if smote_k < 1:
        print(f"  Fold {fold_idx}: SMOTE skipped because a class has too few samples.")
        X_tr_sm, y_tr_sm = X_tr_wc, y_tr
    else:
        smote = SMOTE(
            random_state=RANDOM_SEED,
            k_neighbors=smote_k
        )
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_wc, y_tr)

    clf_sm = LogisticRegression(
        max_iter=4000,
        solver="liblinear",
        class_weight="balanced",
        random_state=RANDOM_SEED,
        C=1.0
    )

    clf_sm.fit(X_tr_sm, y_tr_sm)
    val_preds = clf_sm.predict(X_va_wc)

    fold_macro_f1 = f1_score(
        y_va,
        val_preds,
        average="macro",
        zero_division=0
    )

    fold_f1s_smote.append(fold_macro_f1)

class_counts_cv = np.bincount(y_cv)
min_class_count_cv = class_counts_cv[class_counts_cv > 0].min()
smote_k_final = min(5, min_class_count_cv - 1)

if smote_k_final < 1:
    print("Final SMOTE skipped because a class has too few samples.")
    X_cv_sm, y_cv_sm = X_cv_wc, y_cv
else:
    smote_final = SMOTE(
        random_state=RANDOM_SEED,
        k_neighbors=smote_k_final
    )
    X_cv_sm, y_cv_sm = smote_final.fit_resample(X_cv_wc, y_cv)

lr_smote_final = LogisticRegression(
    max_iter=4000,
    solver="liblinear",
    class_weight="balanced",
    random_state=RANDOM_SEED,
    C=1.0
)

lr_smote_final.fit(X_cv_sm, y_cv_sm)
pred_smote = lr_smote_final.predict(X_test_wc)

smote_cv_mean = np.mean(fold_f1s_smote)
smote_cv_std = np.std(fold_f1s_smote)
smote_acc = accuracy_score(y_test, pred_smote)

smote_f1 = f1_score(
    y_test,
    pred_smote,
    average="macro",
    zero_division=0
)

print(f"  10-Fold CV Macro-F1 : {smote_cv_mean:.4f} ± {smote_cv_std:.4f}")
print(f"  Test Acc            : {smote_acc:.4f}")
print(f"  Test Macro-F1       : {smote_f1:.4f}")

print(
    classification_report(
        y_test,
        pred_smote,
        target_names=["no", "maybe", "yes"],
        zero_division=0
    )
)

axis2_results["A2d: SMOTE+LR w+c"] = {
    "CV Macro-F1": f"{smote_cv_mean:.4f} ± {smote_cv_std:.4f}",
    "Test Acc": smote_acc,
    "Test Macro-F1": smote_f1
}

axis2_preds_dict["A2d_SMOTE_LR"] = pred_smote
all_results["A2d: SMOTE+LR w+c"] = axis2_results["A2d: SMOTE+LR w+c"]
all_preds["A2d_SMOTE_LR"] = pred_smote


# A2e: Soft Voting Ensemble (LR + Cal-SVM + NB)
print("\nA2e — Soft Voting Ensemble (LR + Cal-SVM + NB)")

svm_for_ens = CalibratedClassifierCV(
    LinearSVC(
        C=svm_final.best_params_["C"],
        class_weight="balanced",
        random_state=RANDOM_SEED,
        max_iter=10000
    ),
    method="sigmoid",
    cv=3
)

lr_for_ens = LogisticRegression(
    C=lr_final.best_params_["C"],
    max_iter=4000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_SEED
)

nb_for_ens = MultinomialNB(
    alpha=nb_final.best_params_["alpha"]
)

ensemble = VotingClassifier(
    estimators=[
        ("lr", lr_for_ens),
        ("svm", svm_for_ens),
        ("nb", nb_for_ens)
    ],
    voting="soft",
    n_jobs=-1
)

ensemble.fit(X_cv_wc, y_cv)
pred_ensemble = ensemble.predict(X_test_wc)

ens_acc = accuracy_score(y_test, pred_ensemble)

ens_f1 = f1_score(
    y_test,
    pred_ensemble,
    average="macro",
    zero_division=0
)

print(f"  Test Acc            : {ens_acc:.4f}")
print(f"  Test Macro-F1       : {ens_f1:.4f}")

print(
    classification_report(
        y_test,
        pred_ensemble,
        target_names=["no", "maybe", "yes"],
        zero_division=0
    )
)

axis2_results["A2e: Soft Voting Ensemble"] = {
    "CV Macro-F1": np.nan,
    "Test Acc": ens_acc,
    "Test Macro-F1": ens_f1
}

axis2_preds_dict["A2e_Ensemble"] = pred_ensemble
all_results["A2e: Soft Voting Ensemble"] = axis2_results["A2e: Soft Voting Ensemble"]
all_preds["A2e_Ensemble"] = pred_ensemble



## Section 9 Block 3 — Classical Axis 2 summary


In [ ]:
# SECTION 9 — BLOCK 3 — SUMMARY FOR A2a–A2e

axis2_order = [
    "A2a: LR word+char",
    "A2b: Cal-SVM word+char",
    "A2c: MNB word+char",
    "A2d: SMOTE+LR w+c",
    "A2e: Soft Voting Ensemble"
]

axis2_df = pd.DataFrame.from_dict(
    {k: axis2_results[k] for k in axis2_order},
    orient="index"
)
display(axis2_df)

def get_axis2_cv_mean(name):
    cv_val = axis2_results[name]["CV Macro-F1"]
    if isinstance(cv_val, str):
        return float(cv_val.split(" ± ")[0])
    if pd.isna(cv_val):
        return -np.inf
    return float(cv_val)

best_axis2_name = max(axis2_order, key=get_axis2_cv_mean)
print(f"\nBest Axis 2 classical model (chosen by CV, not test): {best_axis2_name}")

axis2_names_list = axis2_order
axis2_cv_vals = []
axis2_test_vals = []

for k in axis2_names_list:
    cv_val = axis2_results[k]["CV Macro-F1"]
    if isinstance(cv_val, str):
        cv_mean = float(cv_val.split(" ± ")[0])
    elif pd.isna(cv_val):
        cv_mean = np.nan
    else:
        cv_mean = float(cv_val)

    axis2_cv_vals.append(cv_mean)
    axis2_test_vals.append(axis2_results[k]["Test Macro-F1"])

short_names = [n.split(": ")[1] if ": " in n else n for n in axis2_names_list]
x = np.arange(len(short_names))
w = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x, axis2_cv_vals, w, label="CV Macro-F1")
plt.bar(x + w, axis2_test_vals, w, label="Test Macro-F1")
plt.xticks(x + w / 2, short_names, rotation=15)
plt.title("Axis 2 (Classical Models): CV vs Test Macro-F1")
plt.ylabel("Macro-F1")
plt.ylim(0, np.nanmax(axis2_test_vals + axis2_cv_vals) + 0.1)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "visualization_9_axis2_cv_vs_test_macro_f1.png"), dpi=300, bbox_inches="tight")
plt.show()

axis2_name_to_pred_key = {
    "A2a: LR word+char": "A2a_LR_wordchar",
    "A2b: Cal-SVM word+char": "A2b_CalSVM_wordchar",
    "A2c: MNB word+char": "A2c_MNB_wordchar",
    "A2d: SMOTE+LR w+c": "A2d_SMOTE_LR",
    "A2e: Soft Voting Ensemble": "A2e_Ensemble"
}

best_a2_key = axis2_name_to_pred_key[best_axis2_name]
best_a2_preds = axis2_preds_dict[best_a2_key]

cm = confusion_matrix(y_test, best_a2_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["no", "maybe", "yes"],
    yticklabels=["no", "maybe", "yes"],
    cmap="Blues"
)
plt.title(f"Best Axis 2 Classical Confusion Matrix: {best_a2_key}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "visualization_10_best_axis2_confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()